In [ ]:
# 02 · Login con falla · saucedemo (PROD)
#
# Intenta iniciar sesión con locked_out_user (misma contraseña que standard_user).
# saucedemo bloquea a ese usuario a propósito: la ejecución termina en [FALLO] con el
# mensaje del sitio y una captura de pantalla.
#
# Ejecuta la celda. Debajo aparece el log con fecha (dd-mm-aaaa), hora y ambiente.
# El log se guarda en reports/logs/<AMBIENTE>/<aaaa>/<mm>/<dd>/ y las capturas en reports/evidencias/.
# Chrome se abre en modo visible (para ocultarlo: variable de entorno HEADLESS=1).

import contextlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # permite importar pages/ y utils/

from pages import InventoryPage, LoginPage
from utils.config import AMBIENTE, BASE_URL, PASSWORD, USERS
from utils.driver_factory import create_driver
from utils.evidence import save_screenshot
from utils.logger import get_logger, setup_execution_log

log_file = setup_execution_log("02_login_con_falla")
log = get_logger()
log.info("===== INICIO | Login con falla =====")
log.info("Ambiente: %s | Sitio: %s", AMBIENTE, BASE_URL)

USUARIO = USERS["locked_out"]

driver = create_driver()
try:
    # 1. Login
    login_page = LoginPage(driver).load()
    login_page.login(USUARIO, PASSWORD)
    if login_page.is_visible(LoginPage.ERROR):
        mensaje = login_page.error_message()
        log.error("[FALLO] Login rechazado para %s -> %s", USUARIO, mensaje)
        raise AssertionError(f"Login rechazado para {USUARIO}: {mensaje}")
    inventory = InventoryPage(driver).wait_until_loaded()
    log.info("[OK] Login exitoso como %s -> página '%s'", USUARIO, inventory.title())

    log.info("RESULTADO: todas las validaciones OK")
except Exception:
    log.exception("RESULTADO: la ejecución falló")
    with contextlib.suppress(Exception):
        save_screenshot(driver, "02_error")
    raise
finally:
    driver.quit()
    log.info("===== FIN | Log guardado en %s =====", log_file)